# Q2. LLM을 활용해서 해당 문장의 유형 분류하는 유형 분류기를 만들어봅시다.

### In this section, I will provide the codes for building the classifier for categorizing customer inquiries (or queries) using LLMs. I have made total of 2 classifier: one that implements ChatGPT-4o-mini, and one that utilizes Llama3.0-kor model. Note that the following jupyter notebook has been runned on Google Colab.

## ============ With GPT_4o_mini ============

In [ ]:
!pip install pydantic openai

In [ ]:
import os
import re
from typing import Dict, Literal

from openai import OpenAI
from google.colab import userdata
from pydantic import BaseModel

api_key = userdata.get("OPENAI_API_KEY")
os.environ["OPENAI_API_KEY"] = api_key

## Label

In [ ]:
LABELS_KO = {
    # Conversation (Static routing)
    "GREET": "인사",
    "GOODBYE": "대화 종료",
    "FEEDBACK_POS": "긍정 피드백/감사",
    "FEEDBACK_NEG": "부정 피드백/불만",
    "HUMAN_HANDOFF": "상담원 연결 요청",

    # Insurance (RAG routing)
    "PRODUCT_OVERVIEW": "보험 상품의 대략적인 개요에 대한 전반적인 설명",
    "PLAN_OPTIONS": "어떤 플랜/특약 옵션이 존재하는지 설명",
    "COVERAGE_BENEFITS": "보장 범위 (어떤 암까지 보장되는지) / 보장 혜택 / 지급 방식 / 암 발병 시 최대 보장 횟수",
    "EXCLUSIONS_LIMITATIONS": "보장하지 않는 손해(면책 사항) / 지급 제한 사항 / 중증도 기준",
    "WAITING_PERIOD": "대기 기간/면책 기간 (보험 가입 후 특정 질병이나 사고에 대해 보장이 시작되기 전까지의 기간)",
    "PREEXISTING_CONDITIONS": "기존에 있던 질병 / 병력에 따른 보험 조건",
    "ELIGIBILITY_UNDERWRITING": "보험 상품 가입 가능 여부 / 심사",
    "PREMIUM_STABILITY_RENEWAL": "보험료 고정/갱신",
    "PRICING_GENERAL": "전반적인 일반 보험료 / 가격",
    "QUOTE_PERSONALIZED": "개인의 현재 증상에 맞는 맞춤 견적",
    "PAYMENT_BILLING": "보험 상품 구매 시 납입 / 결제 / 청구",
    "COORDINATION_OTHER_INSURANCE": "타 보험과의 연계 / 중복",
    "APPLICATION_HOWTO": "보험 상품 가입 / 청약 방법",
    "CLAIM_FILING": "보험금 청구 방법",
    "CLAIM_DOCUMENTS": "청구 서류",
    "CLAIM_STATUS_TIMELINE": "청구 진행/처리 기간",
    "POLICY_CHANGES_CANCEL_RENEW": "계약 변경/해지/갱신",
    "COMPLAINT_DISPUTE": "민원/이의제기",

    # Safety filtering (Static routing)
    "PRIVACY_PII": "개인정보/민감정보 (고유번호나 주소 등)",
    "MEDICAL_DIAGNOSIS_ADVICE": "의료 상담/진단/치료 조언",
    "OUT_OF_SCOPE": "기타/범위 외 (보험과 관련이 거의 없는 질문)", #This should be routed to RAG
}

In [ ]:
LabelKey = Literal[
    "GREET",
    "GOODBYE",
    "FEEDBACK_POS",
    "FEEDBACK_NEG",
    "HUMAN_HANDOFF",
    "PRODUCT_OVERVIEW",
    "PLAN_OPTIONS",
    "COVERAGE_BENEFITS",
    "EXCLUSIONS_LIMITATIONS",
    "WAITING_PERIOD",
    "PREEXISTING_CONDITIONS",
    "ELIGIBILITY_UNDERWRITING",
    "PREMIUM_STABILITY_RENEWAL",
    "PRICING_GENERAL",
    "QUOTE_PERSONALIZED",
    "PAYMENT_BILLING",
    "COORDINATION_OTHER_INSURANCE",
    "APPLICATION_HOWTO",
    "CLAIM_FILING",
    "CLAIM_DOCUMENTS",
    "CLAIM_STATUS_TIMELINE",
    "POLICY_CHANGES_CANCEL_RENEW",
    "COMPLAINT_DISPUTE",
    "PRIVACY_PII",
    "MEDICAL_DIAGNOSIS_ADVICE",
    "OUT_OF_SCOPE",
]

## Prompt Engineering

In [ ]:
class QueryClassification(BaseModel):
    label: LabelKey
    reason_ko: str


def build_system_prompt(labels_ko):
    label_lines = "\n".join([f"- {k}: {v}" for k, v in labels_ko.items()])

    return f"""너는 한국어 사용자 질의를 아래 라벨 중 정확히 하나로만 분류하는 라우터다.
            반드시 스키마에 맞춰 JSON으로만 응답하여라.

            [라벨 목록]
            {label_lines}

            [분류 규칙(중요)]
            - 대화 행위(인사/종료/피드백/상담원연결)는 보험 내용보다 우선한다.
            - 사용자가 주민번호/전화번호/주소/계좌/카드번호 등 개인정보를 제공하거나 요청하면: PRIVACY_PII
            - 진단/치료/약 복용/검사 해석 등 의료 조언 요청이면: MEDICAL_DIAGNOSIS_ADVICE
            - 보험료가 '고정인지/갱신형인지/갱신 시 인상'처럼 구조를 묻는다면: PREMIUM_STABILITY_RENEWAL
            - 그냥 "얼마에요/보험료/가격"이면: PRICING_GENERAL
            - 개인 정보(나이/성별/직업/병력 등)를 넣어 내 견적을 내달라: QUOTE_PERSONALIZED
            - 보장/급부/지급 조건: COVERAGE_BENEFITS
            - 면책/제한/불지급: EXCLUSIONS_LIMITATIONS
            - 대기기간/면책기간: WAITING_PERIOD
            - 기존 질병/병력으로 가입 가능?: PREEXISTING_CONDITIONS 또는 ELIGIBILITY_UNDERWRITING(심사/인수/가입가능여부가 중심이면)
            - 청구 방법: CLAIM_FILING / 청구서류: CLAIM_DOCUMENTS / 처리기간/진행상태: CLAIM_STATUS_TIMELINE
            - 계약 변경/해지/갱신/철회: POLICY_CHANGES_CANCEL_RENEW
            - 민원/이의제기/분쟁: COMPLAINT_DISPUTE
            - 위 범주로 보기 어렵거나 보험과 무관하면: OUT_OF_SCOPE

            [예시]
            - "안녕하세요" -> GREET
            - "상담원 연결해줘" -> HUMAN_HANDOFF
            - "보험료 갱신형이야? 오르나?" -> PREMIUM_STABILITY_RENEWAL
            - "월 보험료 얼마야?" -> PRICING_GENERAL
            - "7세 남자아이에게 암보험 견적" -> QUOTE_PERSONALIZED
            - "암 진단받았는데 치료 어떻게?" -> MEDICAL_DIAGNOSIS_ADVICE
            """


## Local control: For efficiency

### I have grouped the customer queries into two situations: whether the response should be answered by document retrievals & searching, or whether the response does not require retrievals & searching and be answered in a fixed format - this would reduce the overall computational cost and lead to the chatbot being more efficient.

In [ ]:
RE_SSN = re.compile(r"\b\d{{6}}-\d{{7}}\b")  # 주민등록번호
RE_PHONE = re.compile(r"\b01[016789]-?\d{3,4}-?\d{4}\b") #전화번호
RE_CARD = re.compile(r"\b(?:\d[ -]*?){13,19}\b")  # 카드번호

def strip_spaces(text):
    return re.sub(r"\s+", " ", text.strip())

def heuristic_label(query):
    q = strip_spaces(query).lower()

    if RE_SSN.search(q) or RE_PHONE.search(q):
        return "PRIVACY_PII"
    elif any(k in q for k in ["주민등록", "주민번호", "계좌", "카드번호",
                            "신용카드", "비밀번호", "otp", "주소",
                            "전화번호", "메일주소", "이메일"]):
        return "PRIVACY_PII"
    elif "카드" in q and RE_CARD.search(q):
        return "PRIVACY_PII"

    # 인사/종료/피드백/상담원 연결
    elif any(k in q for k in ["상담원", "상담사", "직원", "사람 상담",
                            "전화 연결", "콜센터", "연결해줘", "연결해 주세요"]):
        return "HUMAN_HANDOFF"
    elif any(k in q for k in ["안녕", "안녕하세요", "안녕!", "안녕?",
                            "반가워", "좋은 아침", "좋은저녁", "hi", "hello"]):
        return "GREET"
    elif any(k in q for k in ["대화 종료", "종료", "종료할게", "끝낼게",
                            "그만", "bye", "잘가", "안녕히", "수고", "나갈게"]):
        return "GOODBYE"

    # -- Test Result: especially for positive feedback, it detects some words
    # -- that should be classified to other category ex: MEDICAL_DIAGNOSIS_ADVICE
    # --> 어떤 상품이 좋을까요? : classified as FEEDBACK_POS but should be classified as MEDICAL_DIAGNOSIS_ADVICE

    # if any(k in q for k in ["감사", "고마워", "덕분", "도움됐", "좋아요", "최고", "고맙습니다"]):
    #     return "FEEDBACK_POS"

    # if any(k in q for k in ["별로", "실망", "불만", "화나", "짜증", "틀렸", "엉망", "최악", "왜이래"]):
    #     return "FEEDBACK_NEG"
    else:
        return None


## LLM Pipeline

In [ ]:
class LLMQueryClassifier:

    def __init__(self, model="gpt-4o-mini-2024-07-18"):
        self.client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        self.model = model
        self.system_prompt = build_system_prompt(LABELS_KO)

    def classify(self, query):
        query = strip_spaces(query)
        if not query:
            return {"label": "OUT_OF_SCOPE", "label_ko": LABELS_KO["OUT_OF_SCOPE"], "reason": "빈 입력"}

        h = heuristic_label(query)

        if h is not None:
            return {"label": h, "label_ko": LABELS_KO[h], "reason": "로컬 규칙에 의해 분류"}

        # LLM
        try:
            res = self.client.responses.parse(
                model=self.model,
                input=[
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": query},
                ],
                text_format=QueryClassification,
                store=False,
                temperature=0,
            )

            parsed: QueryClassification = res.output_parsed

            return {
                "label": parsed.label,
                "label_ko": LABELS_KO[parsed.label],
                "reason": parsed.reason_ko.strip(),
            }

        except Exception as e: # For any functional errors
            return {
                "label": "OUT_OF_SCOPE",
                "label_ko": LABELS_KO["OUT_OF_SCOPE"],
                "reason": f"분류 중 오류. OUT_OF_SCOPE로 처리",
            }



## Demo Run

In [ ]:
if __name__ == "__main__":
    clf = LLMQueryClassifier(model=os.getenv("OPENAI_MODEL", "gpt-4o-mini-2024-07-18"))

    tests = [
        "안녕하세요!",
        "채팅 종료할게",
        "상담원 연결해 주세요.",
        "한화 100세 암보험이 어떤건지 간략하게 알려줘",
        "이 상품 보장 내용이 뭐예요?",
        "어떤 상황에서 보험 적용이 안돼거나 제한되나요?",
        "암 발병 시 최대 몇 회까지 보장이 돼나요?"
        "해당 보험 상품에 면책 기간이 있나요?",
        "한화 100세 암치료보장보험은 갱신형인가요? 아니면 나이를 먹으면 보험료가 오르나요?",
        "한화 100세 암치료보장보험 월 보험료 얼마인가요?",
        "97년생 남자고 비흡연자인데 암보험 견적 받고 싶어요",
        "보험금 청구는 어떻게 해요?",
        "너의 답변이 정보가 부족해",
        "좋은 답변 고마워!",
        "가입을 위해서 필요한 청구 서류는 어떤게 있나요?",
        "처리 기간이 얼마나 걸려요?",
        "계약 해지하고 싶은데 절차 알려줘",
        "재거 현재 배가 아픈데 어떤 치료를 추천해주나요?",
        "제 이메일 주소가 abc123@hanhwa.com 인데 해당 주소로 청구서 보내줘.",
        "아 배고프고 졸려",
    ]

    for q in tests:
        out = clf.classify(q)
        print(f"Q: {q}")
        print(f" -> {out['label']} ({out['label_ko']}) | {out['reason']}")
        print("-" * 60)


Q: 안녕하세요!
 -> GREET (인사) | 로컬 규칙에 의해 분류
------------------------------------------------------------
Q: 채팅 종료할게
 -> GOODBYE (대화 종료) | 로컬 규칙에 의해 분류
------------------------------------------------------------
Q: 상담원 연결해 주세요.
 -> HUMAN_HANDOFF (상담원 연결 요청) | 로컬 규칙에 의해 분류
------------------------------------------------------------
Q: 한화 100세 암보험이 어떤건지 간략하게 알려줘
 -> PRODUCT_OVERVIEW (보험 상품의 대략적인 개요에 대한 전반적인 설명) | 한화 100세 암보험에 대한 전반적인 개요를 요청함.
------------------------------------------------------------
Q: 이 상품 보장 내용이 뭐예요?
 -> COVERAGE_BENEFITS (보장 범위 (어떤 암까지 보장되는지) / 보장 혜택 / 지급 방식 / 암 발병 시 최대 보장 횟수) | 보장 내용에 대한 질문이므로 보장 범위에 해당합니다.
------------------------------------------------------------
Q: 어떤 상황에서 보험 적용이 안돼거나 제한되나요?
 -> EXCLUSIONS_LIMITATIONS (보장하지 않는 손해(면책 사항) / 지급 제한 사항 / 중증도 기준) | 보험 적용이 안되거나 제한되는 상황에 대한 질문입니다.
------------------------------------------------------------
Q: 암 발병 시 최대 몇 회까지 보장이 돼나요?해당 보험 상품에 면책 기간이 있나요?
 -> COVERAGE_BENEFITS (보장 범위 (어떤 암까지 보장되는지) / 보장 혜택 / 지급 방식 / 암 발

# ======== With Llama-3-Korean-Bllossom-8B (Improved) ========
### Two improvements are applied to the original Llama pipeline:
### 1. **Few-Shot Chain-of-Thought (CoT) Prompting** — worked examples with explicit reasoning chains are added to the system prompt to help the model distinguish ambiguous label boundaries (e.g. PRODUCT_OVERVIEW vs COVERAGE_BENEFITS, FEEDBACK_NEG vs OUT_OF_SCOPE).
### 2. **Constrained Decoding via `outlines`** — the `outlines` library replaces manual JSON parsing. It modifies sampling logits at every token step so that the output is *guaranteed* to be valid JSON conforming to the `QueryClassification` Pydantic schema, eliminating parse failures and invalid-label hallucinations entirely. (Willard & Louf, 2023, arXiv:2307.09702)

In [ ]:
!pip install -U accelerate pydantic==2.12.3 outlines

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.4/103.4 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 105.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 7.6 MB/s eta 0:00:00


In [ ]:
import huggingface_hub
huggingface_hub.login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import json
import os
import re
from typing import Literal

import outlines
import torch
from pydantic import BaseModel, ValidationError
from transformers import AutoModelForCausalLM, AutoTokenizer

In [ ]:
LABELS_KO = {
    # Conversation (Static routing)
    "GREET": "인사",
    "GOODBYE": "대화 종료",
    "FEEDBACK_POS": "긍정 피드백/감사",
    "FEEDBACK_NEG": "부정 피드백/불만",
    "HUMAN_HANDOFF": "상담원 연결 요청",

    # Insurance (RAG routing)
    "PRODUCT_OVERVIEW": "보험 상품의 대략적인 개요에 대한 전반적인 설명",
    "PLAN_OPTIONS": "어떤 플랜/특약 옵션이 존재하는지 설명",
    "COVERAGE_BENEFITS": "보장 범위 (어떤 암까지 보장되는지) / 보장 혜택 / 지급 방식 / 암 발병 시 최대 보장 횟수",
    "EXCLUSIONS_LIMITATIONS": "보장하지 않는 손해(면책 사항) / 지급 제한 사항 / 중증도 기준",
    "WAITING_PERIOD": "대기 기간/면책 기간 (보험 가입 후 특정 질병이나 사고에 대해 보장이 시작되기 전까지의 기간)",
    "PREEXISTING_CONDITIONS": "기존에 있던 질병 / 병력에 따른 보험 조건",
    "ELIGIBILITY_UNDERWRITING": "보험 상품 가입 가능 여부 / 심사",
    "PREMIUM_STABILITY_RENEWAL": "보험료 고정/갱신",
    "PRICING_GENERAL": "전반적인 일반 보험료 / 가격",
    "QUOTE_PERSONALIZED": "개인의 현재 증상에 맞는 맞춤 견적",
    "PAYMENT_BILLING": "보험 상품 구매 시 납입 / 결제 / 청구",
    "COORDINATION_OTHER_INSURANCE": "타 보험과의 연계 / 중복",
    "APPLICATION_HOWTO": "보험 상품 가입 / 청약 방법",
    "CLAIM_FILING": "보험금 청구 방법",
    "CLAIM_DOCUMENTS": "청구 서류",
    "CLAIM_STATUS_TIMELINE": "청구 진행/처리 기간",
    "POLICY_CHANGES_CANCEL_RENEW": "계약 변경/해지/갱신",
    "COMPLAINT_DISPUTE": "민원/이의제기",

    # Safety filtering (Static routing)
    "PRIVACY_PII": "개인정보/민감정보 (고유번호나 주소 등)",
    "MEDICAL_DIAGNOSIS_ADVICE": "의료 상담/진단/치료 조언",
    "OUT_OF_SCOPE": "기타/범위 외 (보험과 관련이 거의 없는 질문)", #This should be routed to RAG
}

In [ ]:
LabelKey = Literal[
    "GREET",
    "GOODBYE",
    "FEEDBACK_POS",
    "FEEDBACK_NEG",
    "HUMAN_HANDOFF",
    "PRODUCT_OVERVIEW",
    "PLAN_OPTIONS",
    "COVERAGE_BENEFITS",
    "EXCLUSIONS_LIMITATIONS",
    "WAITING_PERIOD",
    "PREEXISTING_CONDITIONS",
    "ELIGIBILITY_UNDERWRITING",
    "PREMIUM_STABILITY_RENEWAL",
    "PRICING_GENERAL",
    "QUOTE_PERSONALIZED",
    "PAYMENT_BILLING",
    "COORDINATION_OTHER_INSURANCE",
    "APPLICATION_HOWTO",
    "CLAIM_FILING",
    "CLAIM_DOCUMENTS",
    "CLAIM_STATUS_TIMELINE",
    "POLICY_CHANGES_CANCEL_RENEW",
    "COMPLAINT_DISPUTE",
    "PRIVACY_PII",
    "MEDICAL_DIAGNOSIS_ADVICE",
    "OUT_OF_SCOPE",
]



In [ ]:
class QueryClassification(BaseModel):
    label: LabelKey
    reason_ko: str


def build_system_prompt(labels_ko):
    label_lines = "\n".join([f"- {k}: {v}" for k, v in labels_ko.items()])

    return f"""너는 한국어 사용자 질의를 아래 라벨 중 '정확히 하나'로 분류하는 라우터다.
반드시 아래 JSON 스키마로만 출력하고, JSON 외의 텍스트는 절대 출력하지 마라.

[출력 JSON 스키마]
{{"label": "<LABEL_KEY>", "reason_ko": "<짧은 근거 1~2문장>"}}

[라벨 목록]
{label_lines}

[분류 규칙(중요)]
- 대화 행위(인사/종료/피드백/상담원연결)는 보험 내용보다 우선한다.
- 사용자가 주민번호/전화번호/주소/계좌/카드번호 등 개인정보를 제공하거나 요청하면: PRIVACY_PII
- 진단/치료/약 복용/검사 해석 등 의료 조언 요청이면: MEDICAL_DIAGNOSIS_ADVICE
- 보험료가 '고정인지/갱신형인지/갱신 시 인상'처럼 구조를 묻는다면: PREMIUM_STABILITY_RENEWAL
- 그냥 "얼마에요/보험료/가격"이면: PRICING_GENERAL
- 개인 정보(나이/성별/직업/병력 등)를 넣어 내 견적을 내달라: QUOTE_PERSONALIZED
- 보장/급부/지급 조건: COVERAGE_BENEFITS
- 면책/제한/불지급: EXCLUSIONS_LIMITATIONS
- 대기기간/면책기간: WAITING_PERIOD
- 기존 질병/병력으로 가입 가능?: PREEXISTING_CONDITIONS 또는 ELIGIBILITY_UNDERWRITING(심사/인수/가입가능여부가 중심이면)
- 청구 방법: CLAIM_FILING / 청구서류: CLAIM_DOCUMENTS / 처리기간/진행상태: CLAIM_STATUS_TIMELINE
- 계약 변경/해지/갱신/철회: POLICY_CHANGES_CANCEL_RENEW
- 민원/이의제기/분쟁: COMPLAINT_DISPUTE
- 챗봇/상담봇의 답변 품질에 대한 불만/비판이면: FEEDBACK_NEG
- 챗봇/상담봇의 답변에 대한 감사/칭찬이면: FEEDBACK_POS
- 보험과도 무관하고 챗봇 피드백도 아닌 완전한 일상 잡담이면: OUT_OF_SCOPE

[핵심 구별 기준]
▶ PRODUCT_OVERVIEW vs COVERAGE_BENEFITS
  - "어떤 상품인지", "개요", "간략하게 알려줘", "소개해줘" → PRODUCT_OVERVIEW (상품 전체 개요)
  - "보장 내용", "무엇을 보장", "어떤 혜택", "지급 조건" → COVERAGE_BENEFITS (구체적 보장 내용)
▶ FEEDBACK_NEG vs OUT_OF_SCOPE
  - "답변이 부족해", "설명이 이상해", "별로야" (챗봇에 대한 평가) → FEEDBACK_NEG
  - "배고프고 졸려", "오늘 날씨 어때" (보험·챗봇과 완전 무관한 잡담) → OUT_OF_SCOPE

[추론 예시 — Chain-of-Thought]

Q: 이 보험 상품이 뭔지 대략적으로 설명해줄 수 있어?
추론: "뭔지 대략적으로 설명해줘"는 상품의 전반적인 소개를 요청하는 표현이다.
  특정 보장 항목이나 혜택을 묻는 것이 아니라 상품 자체가 무엇인지 알고 싶은 것이다.
→ {{"label": "PRODUCT_OVERVIEW", "reason_ko": "상품 전체에 대한 개략적 설명 요청이므로 PRODUCT_OVERVIEW로 분류"}}

Q: 암 진단 시 보험금이 어떤 방식으로 지급되나요?
추론: "보험금 지급 방식"은 보장의 구체적인 내용을 묻고 있다. 상품 개요가 아니라 급부 조건에 해당한다.
→ {{"label": "COVERAGE_BENEFITS", "reason_ko": "보험금 지급 방식이라는 구체적 보장 내용에 대한 질문이므로 COVERAGE_BENEFITS로 분류"}}

Q: 방금 알려준 내용이 너무 짧아서 이해가 안 돼
추론: 사용자가 챗봇이 제공한 답변의 품질에 대해 불만을 표현하고 있다.
  보험 상품에 대한 질문이 아니라 대화 상대(챗봇)의 응답에 대한 부정적 평가다.
→ {{"label": "FEEDBACK_NEG", "reason_ko": "챗봇 답변의 품질에 대한 부정적 피드백이므로 FEEDBACK_NEG로 분류"}}

Q: 오늘 점심 뭐 먹지
추론: 보험과 전혀 관련이 없고, 챗봇의 답변에 대한 피드백도 아닌 완전한 일상 대화다.
→ {{"label": "OUT_OF_SCOPE", "reason_ko": "보험 및 챗봇 피드백과 무관한 일상 잡담이므로 OUT_OF_SCOPE로 분류"}}

Q: 30대 여성 직장인인데 이 보험 가입하면 월 얼마 정도 나오나요?
추론: 나이대(30대), 성별(여성), 직업(직장인)이라는 개인 정보를 제공하며 본인에게 해당하는 보험료를 묻고 있다.
→ {{"label": "QUOTE_PERSONALIZED", "reason_ko": "개인 정보를 제시한 맞춤 보험료 문의이므로 QUOTE_PERSONALIZED로 분류"}}

Q: 가입 후에 보장 플랜을 변경하거나 중도 해약할 수 있나요?
추론: 계약 이후의 플랜 변경과 해약 가능 여부를 묻고 있다. 이는 계약 변경/해지 범주에 해당한다.
→ {{"label": "POLICY_CHANGES_CANCEL_RENEW", "reason_ko": "계약 변경 및 해약 가능 여부 문의이므로 POLICY_CHANGES_CANCEL_RENEW로 분류"}}
"""


## Local control: For efficiency

In [ ]:
RE_SSN = re.compile(r"\b\d{6}-\d{7}\b")  # 주민등록번호
RE_PHONE = re.compile(r"\b01[016789]-?\d{3,4}-?\d{4}\b")
RE_CARD = re.compile(r"\b(?:\d[ -]*?){13,19}\b")  # 카드/긴 숫자열

def strip_space(text):
    return re.sub(r"\s+", " ", text.strip())

def heuristic_label(query):
    q = strip_space(query).lower()

    if RE_SSN.search(q) or RE_PHONE.search(q):
        return "PRIVACY_PII"
    elif any(k in q for k in ["주민등록", "주민번호", "계좌", "카드번호", "신용카드", "비밀번호", "otp", "주소", "전화번호", "메일주소", "이메일"]):
        return "PRIVACY_PII"
    elif "카드" in q and RE_CARD.search(q):
        return "PRIVACY_PII"

    # 상담원 연결/인사/종료/피드백
    elif any(k in q for k in ["안녕하세요", "안녕!", "안녕?", "반가워", "좋은 아침", "좋은저녁", "hi", "hello"]):
        return "GREET"
    elif any(k in q for k in ["대화 종료", "종료할게", "끝낼게", "그만", "bye", "잘가", "안녕히", "수고", "나갈게"]):
        return "GOODBYE"
    elif any(k in q for k in ["상담원", "상담사", "직원", "사람 상담", "전화 연결", "콜센터", "연결해줘", "연결해 주세요"]):
        return "HUMAN_HANDOFF"
    elif any(k in q for k in ["감사", "고마워", "덕분", "도움됐", "좋아요", "최고", "고맙습니다"]):
        return "FEEDBACK_POS"
    elif any(k in q for k in ["별로", "실망", "불만", "화나", "짜증", "틀렸", "엉망", "최악", "왜이래"]):
        return "FEEDBACK_NEG"

    else:
        return None

## Constrained Decoding with `outlines`

### Instead of free-form generation followed by regex-based JSON parsing, we use the `outlines` library to apply *constrained decoding* at the token level.
### At every generation step, `outlines` modifies the logit distribution so that only tokens consistent with the target JSON schema (and valid `LabelKey` strings) have non-zero probability.
### This means the output is **guaranteed** to be a valid `QueryClassification` object

## LLM Pipeline

In [ ]:
class LlamaQueryClassifier:
    def __init__(
        self,
        model_name="MLP-KTLim/llama-3-Korean-Bllossom-8B",
        max_new_tokens=256,
    ):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.system_prompt = build_system_prompt(LABELS_KO)

        # Tokenizer for chat template formatting
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
        if self.tokenizer.pad_token_id is None and self.tokenizer.eos_token_id is not None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # Load HuggingFace model
        hf_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

        # Wrap with outlines for constrained decoding
        self.outlines_model = outlines.from_transformers(hf_model, self.tokenizer)

        # outlines 1.2.13: outlines.generate was removed entirely.
        # The new API is outlines.Generator(model, outlines.json_schema(Schema)).
        # Both Generator and json_schema are exported directly from outlines.__init__.
        self.generator = outlines.Generator(
            self.outlines_model,
            outlines.json_schema(QueryClassification),
        )

    def format_prompt(self, user_query):
        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": user_query},
        ]
        if hasattr(self.tokenizer, "apply_chat_template"):
            return self.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        return f"[SYSTEM]\n{self.system_prompt}\n\n[USER]\n{user_query}\n\n[ASSISTANT]\n"

    def llm_classify(self, query):
        prompt = self.format_prompt(query)
        try:
            raw = self.generator(prompt, max_new_tokens=self.max_new_tokens)

            # outlines 1.2.13 returns a schema-valid JSON string, not a Pydantic object.
            # Parse it manually into QueryClassification.
            if isinstance(raw, str):
                parsed = QueryClassification.model_validate(json.loads(raw))
            else:
                # Already a dict or Pydantic object (defensive, for future API changes)
                parsed = QueryClassification.model_validate(raw)

            return {
                "label": parsed.label,
                "label_ko": LABELS_KO[parsed.label],
                "reason": parsed.reason_ko.strip(),
            }
        except Exception as e:
            return {
                "label": "OUT_OF_SCOPE",
                "label_ko": LABELS_KO["OUT_OF_SCOPE"],
                "reason": f"분류 중 오류. OUT_OF_SCOPE로 처리: {e}",
            }

    def classify(self, query):
        query = strip_space(query)
        if not query:
            return {"label": "OUT_OF_SCOPE", "label_ko": LABELS_KO["OUT_OF_SCOPE"], "reason": "빈 입력"}

        h = heuristic_label(query)
        if h is not None:
            return {"label": h, "label_ko": LABELS_KO[h], "reason": "로컬 규칙에 의해 분류"}

        return self.llm_classify(query)

## Demo Run

In [ ]:
if __name__ == "__main__":

    clf = LlamaQueryClassifier(
        model_name="MLP-KTLim/llama-3-Korean-Bllossom-8B",
        max_new_tokens=128,
    )

    tests = [
        "안녕하세요!",
        "채팅 종료할게",
        "상담원 연결해 주세요.",
        "한화 100세 암보험이 어떤건지 간략하게 알려줘",
        "이 상품 보장 내용이 뭐예요?",
        "어떤 상황에서 보험 적용이 안돼거나 제한되나요?",
        "암 발병 시 최대 몇 회까지 보장이 돼나요?",
        "해당 보험 상품에 면책 기간이 있나요?",
        "한화 100세 암치료보장보험은 갱신형인가요? 아니면 나이를 먹으면 보험료가 오르나요?",
        "한화 100세 암치료보장보험 월 보험료 얼마인가요?",
        "97년생 남자고 비흡연자인데 암보험 견적 받고 싶어요",
        "보험금 청구는 어떻게 해요?",
        "너의 답변이 정보가 부족해",
        "좋은 답변 고마워!",
        "가입을 위해서 필요한 청구 서류는 어떤게 있나요?",
        "처리 기간이 얼마나 걸려요?",
        "계약 해지하고 싶은데 절차 알려줘",
        "재거 현재 배가 아픈데 어떤 치료를 추천해주나요?",
        "제 이메일 주소가 abc123@hanhwa.com 인데 해당 주소로 청구서 보내줘.",
        "아 배고프고 졸려",
    ]

    for q in tests:
        out = clf.classify(q)
        print(f"Q: {q}")
        print(f" -> {out['label']} ({out['label_ko']}) | {out['reason']}")
        print("-" * 60)


config.json:   0%|          | 0.00/710 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

Q: 처리 기간이 얼마나 걸려요?
 -> CLAIM_STATUS_TIMELINE (청구 진행/처리 기간) | 처리 기간에 대한 문의로, 보험금 청구와 관련된 처리 시간에 대한 정보를 제공하므로 CLAIM_STATUS_TIMELINE으로 분류
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: 계약 해지하고 싶은데 절차 알려줘
 -> CLAIM_FILING (보험금 청구 방법) | 계약 해지 절차를 묻는 것이므로 CLAIM_FILING로 분류
------------------------------------------------------------


Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


Q: 재거 현재 배가 아픈데 어떤 치료를 추천해주나요?
 -> MEDICAL_DIAGNOSIS_ADVICE (의료 상담/진단/치료 조언) | 의료 상담/진단/치료 조언 요청이므로 MEDICAL_DIAGNOSIS_ADVICE로 분류
------------------------------------------------------------
Q: 제 이메일 주소가 abc123@hanhwa.com 인데 해당 주소로 청구서 보내줘.
 -> PRIVACY_PII (개인정보/민감정보 (고유번호나 주소 등)) | 로컬 규칙에 의해 분류
------------------------------------------------------------
Q: 아 배고프고 졸려
 -> OUT_OF_SCOPE (기타/범위 외 (보험과 관련이 거의 없는 질문)) | 보험 및 챗봇 피드백과 무관한 일상적인 감정 표현이므로 OUT_OF_SCOPE로 분류
------------------------------------------------------------
